# مهام اليوم الثاني - الأسبوع الرابع (أسبوع المشاريع)
تطبيق عملي للأقسام 5 و 6 و 7.

## القسم 5: التنظيف + سجل القرارات (Decision Log)

In [ ]:
import os
import numpy as np
import pandas as pd

os.makedirs("data", exist_ok=True)

# قراءة ملف البيانات الأولي (غيّر المسار إلى اسم ملفك الفعلي إذا اختلف)
df = pd.read_csv("data/raw.csv")

# تحديد عمود الهدف (Target) وعدم استخدامه في تعبئة أي عمود آخر
target_col = "profit" if "profit" in df.columns else df.columns[-1]

decision_log = []

# 1. معالجة عمود رقمي فيه نقص باستخدام الوسيط median بدون لمس الـ target
num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != target_col]
if num_cols:
    num_col = "sales" if "sales" in num_cols else num_cols[0]
    missing_num = int(df[num_col].isna().sum())
    missing_num_pct = round((missing_num / len(df)) * 100, 2)

    if missing_num > 0:
        median_val = df[num_col].median()
        df[num_col] = df[num_col].fillna(median_val)
        decision_log.append({
            "المشكلة": f"نقص قيم في العمود الرقمي {num_col}",
            "القرار": f"تعبئة القيم المفقودة بالوسيط ({median_val:.2f}) دون استخدام {target_col}",
            "السبب بالرقم": f"عدد الصفوف المتأثرة {missing_num} بنسبة {missing_num_pct}%",
            "الصفوف المتأثرة": missing_num,
        })

# 2. معالجة عمود نصي فيه نقص مع حفظ معلومة موجود/غير موجود
cat_cols = [c for c in df.select_dtypes(include=["object", "string"]).columns if c != target_col]
if cat_cols:
    cat_col = cat_cols[0]
    missing_cat = int(df[cat_col].isna().sum())
    missing_cat_pct = round((missing_cat / len(df)) * 100, 2)
    df[f"{cat_col}_is_present"] = df[cat_col].notna().astype(int)
    df[cat_col] = df[cat_col].fillna("غير محدد")
    decision_log.append({
        "المشكلة": f"نقص قيم في العمود النصي {cat_col}",
        "القرار": f"إضافة عمود {cat_col}_is_present وتعويض النقص بـ 'غير محدد'",
        "السبب بالرقم": f"عدد الصفوف المتأثرة {missing_cat} بنسبة {missing_cat_pct}%",
        "الصفوف المتأثرة": missing_cat,
    })

# حفظ النسخة النظيفة
df.to_csv("data/clean.csv", index=False)

# عرض جدول Decision Log
log_df = pd.DataFrame(decision_log)
display(log_df) if not log_df.empty else print("لا توجد قيم مفقودة في الأعمدة المفحوصة.")

## القسم 6: هندسة الميزات (Feature Engineering)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv("data/clean.csv")
target_col = "profit" if "profit" in df.columns else df.columns[-1]

num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c != target_col]
sales_col = "sales" if "sales" in num_cols else num_cols[0]
order_col = "orders" if "orders" in num_cols and "orders" != sales_col else sales_col

# 1. ميزة جديدة: نسبة الهدف إلى المبيعات (هامش الربح)
df["profit_ratio"] = np.where(df[sales_col] != 0, df[target_col] / df[sales_col], 0.0)

# 2. ميزة جديدة: فئة حجم المبيعات (منخفض / مرتفع)
median_sales = df[sales_col].median()
df["high_sales_flag"] = (df[sales_col] >= median_sales).astype(int)

# 3. ميزة جديدة: مؤشر النشاط
df["activity_index"] = df[sales_col] * df[order_col]

new_features = [
    ("profit_ratio", "أضفته لقياس كفاءة العائد مقارنة بحجم المبيعات"),
    ("high_sales_flag", "أضفته للتمييز بين العمليات الأعلى والأقل من الوسيط"),
    ("activity_index", "أضفته لدمج حجم النشاط مع القيمة المالية"),
]

for col, reason in new_features:
    plt.figure(figsize=(6, 4))
    if df[col].nunique() <= 5:
        grouped = df.groupby(col)[target_col].mean()
        grouped.plot(kind="bar")
        plt.ylabel(f"متوسط {target_col}")
        diff_val = grouped.max() - grouped.min() if len(grouped) > 1 else grouped.iloc[0]
        stat_sentence = f"الفرق بين أعلى وأقل متوسط لـ {target_col} عبر فئات {col} يساوي {diff_val:.2f}."
    else:
        plt.scatter(df[col], df[target_col], alpha=0.6)
        plt.ylabel(target_col)
        corr_val = df[col].corr(df[target_col])
        corr_val = 0.0 if np.isnan(corr_val) else corr_val
        stat_sentence = f"معامل الارتباط بين {col} و {target_col} يساوي {corr_val:.2f}."

    plt.title(f"{col} vs {target_col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()

    print(f"العمود {col}: {reason}.")
    print(stat_sentence)
    print("-" * 50)

df.to_csv("data/clean_features.csv", index=False)

## القسم 7: اختيار الميزات (Feature Selection)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = pd.read_csv("data/clean_features.csv")
target_col = "profit" if "profit" in df.columns else df.columns[-1]

selection_rows = []
cols_to_drop = []

for col in df.columns:
    if col == target_col:
        selection_rows.append({
            "العمود": col,
            "القرار": "إبقاء (Target)",
            "السبب": "العمود الهدف المطلوب التنبؤ به",
            "الرقم": 1.0,
        })
        continue

    top_freq_ratio = float(df[col].value_counts(dropna=False, normalize=True).iloc[0])
    unique_ratio = float(df[col].nunique(dropna=False) / max(len(df), 1))

    if top_freq_ratio > 0.95:
        cols_to_drop.append(col)
        selection_rows.append({
            "العمود": col,
            "القرار": "حذف",
            "السبب": "العمود قيمته واحدة تقريباً ولا يضيف تبايناً مفيداً",
            "الرقم": round(top_freq_ratio, 2),
        })
    elif df[col].dtype == "object" and unique_ratio > 0.90:
        cols_to_drop.append(col)
        selection_rows.append({
            "العمود": col,
            "القرار": "حذف",
            "السبب": "معرف أو نص شبه فريد لا يفيد مباشرة كخاصية عامة",
            "الرقم": round(unique_ratio, 2),
        })
    else:
        selection_rows.append({
            "العمود": col,
            "القرار": "إبقاء",
            "السبب": "يحتفظ بتنوع جيد وقابل للاستخدام في النموذج",
            "الرقم": round(1.0 - top_freq_ratio, 2),
        })

selection_df = pd.DataFrame(selection_rows)
display(selection_df)

# رسمة داعمة لقرار اختيار الميزات
decision_counts = selection_df["القرار"].value_counts()
plt.figure(figsize=(6, 4))
decision_counts.plot(kind="bar")
plt.title("Feature Selection Decisions")
plt.xlabel("Decision")
plt.ylabel("Columns Count")
plt.tight_layout()
plt.show()

# حفظ الملف النهائي الجاهز للنمذجة
model_df = df.drop(columns=cols_to_drop)
model_df.to_csv("data/model_ready.csv", index=False)
print("تم حفظ data/model_ready.csv بعدد أعمدة:", model_df.shape[1])